In [1]:
import os

from dotenv import load_dotenv
from numba.core.compiler_machinery import pass_info
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
import magic

In [2]:
load_dotenv()

hf_api_key = os.getenv("HUGGIN_FACE_API")


In [3]:
client = OpenAI(
    api_key=hf_api_key,
    base_url="https://router.huggingface.co/v1"
)

model_name = "Qwen/Qwen3-4B-Instruct-2507"
vision_model_name = "Qwen/Qwen3-VL-4B-Instruct"


###test#######
response = client.chat.completions.create(
    model= model_name,
    messages= [
        {
            "role":"user", "content":"give me a simple code for rag using langchain"
        }
    ]
)

#print(response.choices[0].message.content)
display(Markdown(response.choices[0].message.content))




Sure! Here's a **simple and practical example** of a **Retrieval-Augmented Generation (RAG)** pipeline using **LangChain** in Python. This example:

- Loads documents (e.g., from a text file)
- Indexes them using a simple vector store (ChromaDB)
- Retrieves relevant documents when querying
- Generates a response using an LLM

> 🔧 **Requirements**: Python 3.10+, LangChain, Chroma, OpenAI (or any compatible LLM)

---

### ✅ Step 1: Install the required packages

```bash
pip install langchain langchain-community chromadb openai
```

> (You can use `google.generativeai` or `anthropic` instead of OpenAI if preferred. This example uses OpenAI for simplicity.)

---

### ✅ Step 2: Simple RAG Code

```python
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain import PromptTemplate

# Step 1: Load your document (e.g., a text file)
document_path = "sample.txt"  # Replace with your file path
loader = TextLoader(file_path=document_path)
documents = loader.load()

# Step 2: Split text into chunks (for embedding)
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

# Step 3: Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # or use "text-embedding-ada-002"
vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

# Step 4: Initialize LLM (e.g., GPT-3.5-turbo via OpenAI)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

# Step 5: Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True,
    chain_type="stuff"
)

# Step 6: Ask a question
query = "What is the main idea of the text?"
result = qa_chain.invoke(query)

# Output result
print("Query:", query)
print("Answer:", result["result"])
print("Sources:", [doc.page_content[:100] + "..." for doc in result["source_documents"]])
```

---

### 📝 Example `sample.txt` content:

```
The purpose of this document is to explain how RAG works in LangChain. RAG allows LLMs to access external knowledge by retrieving relevant documents. It combines retrieval with generation to produce accurate and context-aware responses.
```

---

### 🚀 How it works:

- Your text is split into smaller chunks.
- Embeddings are generated and stored in ChromaDB.
- When you ask a question, the system retrieves the most relevant chunk.
- The LLM generates a response based on that context.

---

### 📌 Notes:

- Replace `"sample.txt"` with your own file or data source (PDF, HTML, etc.).
- You can change the LLM to `GoogleGemini`, `Anthropic`, or local models using LangChain's LLMs.
- Chroma is a local vector database — no cloud needed.
- For production, consider adding error handling, cache, or authentication.

---

✅ Done! You now have a **working, simple RAG pipeline** using LangChain.

Let me know if you'd like a version with **PDF support**, **local LLMs (like Llama 3)**, or **more advanced setup**! 🚀

In [4]:
def classify_file(mime, suffix):
    if mime == "application/pdf":
        return "PDF"

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        return "Word"

    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        return "Excel"

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:
        return "Powerpoint"

    if mime.startswith("image/"):
        return "Image"

    if mime.startswith("audio/"):
        return "Audio"

    if mime.startswith("video/"):
        return "Video"

    return "invalid"

file_path = r"C:\Users\shahin\Desktop\computers-15-00292.pdf"

path = Path(file_path)

suffix, mime = None, None

if path.exists():
    suffix = path.suffix
    mime = magic.from_file(str(path), mime=True)


def route_file(mime, suffix, path):
    if mime == "application/pdf":
        #pdf
        handle_pdf(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        #word
        handle_word(path)


    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        handle_excel(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:
        handle_pp()

    # if mime.startswith("image/"):
    #     return "Image"
    #
    # if mime.startswith("audio/"):
    #     return "Audio"
    #
    # if mime.startswith("video/"):
    #     return "Video"

    return "invalid"







###########Alternative using model###################
# print(mime, suffix)
# system_prompt = """
# Your role is just to analyse the MIME and the suffix passed to you and detect the file type.
# You must identify if it is Excel, Word document, Powerpoint, Audio, Video, Image or PDF.
# Just return one word.
# If what is provided to you is not valid just return the word: invalid.
# """
#
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {
#             "role": "user",
#             "content": f"MIME: {mime}, suffix: {suffix}"
#         }
#     ]
# )
#
# print(response.choices[0].message.content)

In [5]:
import

def handle_word(path):
    pass

def handle_excel(path):
    pass

def handle_pp(path):
    pass


def handle_pdf(path):
    pass

**Embeddings models**


1. mulitlangual e5-v2 for sentences
2. open-clip for images


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")